## Tasks
```
Data Cleaning: Remove duplicates, handle missing reviews if any, preprocess text (lowercasing, stopwords removal).
Exploratory Analysis: Word clouds, sentiment distribution, most common positive/negative words.

Model Development: Use NLP techniques (TF-IDF, Word2Vec, or BERT embeddings) with models like Logistic Regression, SVM, or Neural Networks.
Validation: Use train/test split, cross-validation, and metrics like accuracy, F1-score.


Submission Guidelines
•	Your submission should include a comprehensive report and the complete codebase.
•	Your code should be well-documented and include comments explaining the major steps.

Evaluation Criteria
•	Correct implementation of data preprocessing and feature extraction.
•	Accuracy and robustness of the classification model.
•	Depth and insightfulness of the sentiment analysis.
•	Clarity and thoroughness of the evaluation and discussion sections.
•	Overall quality and organization of the report and code.
Good luck, and we look forward to your insightful analysis of the given dataset!
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Loading dataset into our enviroment
dataset = pd.read_csv("amazonreviews.tsv",sep="\t")

# Working on a copy set
df = dataset.copy()

print("\n<------------Counting missing values------------>\n")
print(df.isnull().sum())

print("\n<---------Checking the Uniqueness of the label column----------->\n")
print(df["label"].unique())

print("\n<--------Number of duplicates in the dataset on the basis of the review----------->\n")
print(df["review"].duplicated().sum())

In [ ]:
import nltk, string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize

# data cleaning
df = df.drop_duplicates().reset_index(drop=True)
df= df.dropna(subset=['review','label']).reset_index(drop=True)

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text_lib(text):
   
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t.isalpha() and t not in stop] # if t is alphanumeric add else not 
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)


df['clean_review'] = df['review'].astype(str).apply(clean_text_lib) # Creating a new column with clean_review

print(df)

## Exploratory Data Analysis(EDA)

In [ ]:
from wordcloud import WordCloud
from collections import Counter


print(df['label'].value_counts(normalize=True))
sns.countplot(x='label',data=df)
plt.title("Label Counts")
plt.show()

def show_wordcloud(text,title):
    wc = WordCloud(width=800, height=400).generate(" ".join(text))
    plt.figure(figsize=(15,10))
    plt.imshow(wc,interpolation='bilinear')
    plt.axis("off")
    if title:
        plt.title(title)
    
    plt.show()


show_wordcloud(df['clean_review'],"All words")
show_wordcloud(df[df['label'] == 'pos']['clean_review'], "All positive reviws words cloud")
show_wordcloud(df[df['label'] == 'neg']['clean_review'], "All negative reviews words cloud")

# Most common words finding
def top_n_common(series,n=20):
    countings = Counter(" ".join(series).split())
    return countings.most_common(n)

print(top_n_common(df[df['label'] == 'pos']['clean_review']))
print(top_n_common(df[df['label'] == 'neg']['clean_review']))

pos_words = top_n_common(df[df['label'] == 'pos']['clean_review'])
neg_words = top_n_common(df[df['label'] == 'neg']['clean_review'])


# Plotting the words and their occurence
# Positive word occurence
plt.figure(figsize=(15,10))
sns.barplot(x=[w[1] for w in pos_words],y=[w[0] for w in pos_words])
plt.title("Occurence of positive words")
plt.xticks(rotation=45)
plt.show()

#Negatie Word occurence
plt.figure(figsize=(15,10))
sns.barplot(x=[w[1] for w in neg_words],y=[w[0] for w in neg_words])
plt.title("Occurence of negative words")
plt.xticks(rotation=45)
plt.show()

## 4. Feature Extraction:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X = df['clean_review']
y = df['label'].map({'pos':1,'neg':0})

X_train, X_test,y_train,y_test = train_test_split(X, y, test_size=42, random_state=42, stratify=y)

# Tf-IDF using
tfidf= TfidfVectorizer(max_features = 2000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf= tfidf.transform(X_test)


## Model Building And Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

pipe_lr = Pipeline([('tfidf',tfidf),('clf',LogisticRegression(max_iter=1000,solver='saga'))])

parameters = {
    'clf__penalty':['l2','l1'],
    'clf__C':[0.01,0.1,1,10]
}
lr_grid = GridSearchCV(pipe_lr,parameters,cv=5,scoring='f1')
lr_grid.fit(X_train,y_train)


In [ ]:
from sklearn.metrics import accuracy_score
# Best prameters
print("Best lr parameters:\n",lr_grid.best_params_)

y_pred = lr_grid.predict(X_test)
# Classification report
print("\n<--------Classsification report---------->\n")
print(classification_report(y_pred,y_test))
print("Area Under the curve(AUC)_score:", roc_auc_score(y_test,y_pred))
print("Accuracy score for LR:",accuracy_score(y_test,lr_grid.predict(X_test)))

In [ ]:
# SVM - linear svc
from sklearn.svm import LinearSVC

pipe_svm = Pipeline([('tfidf',tfidf),('clf',LinearSVC())])

params_svm = {
    'clf__C':[0.01,0.1,1,1.0]
}
svm_grid = GridSearchCV(pipe_svm,params_svm,cv=5,scoring='f1')
svm_grid.fit(X_train, y_train)


In [ ]:
# Scoring for the svm model
from sklearn.metrics import accuracy_score
print("Best SVM Parameters:\n",svm_grid.best_params_)
y_pred = svm_grid.predict(X_test)
print("Classification report:\n",classification_report(y_test,y_pred))
print("Accuracy Score for svm:", accuracy_score(y_test,y_pred))

In [ ]:
# Neural Network/Simple Keras Model 
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(X_train_tfidf.shape[1],)),
    layers.Dense(128,activation='relu'),   # here  128 is neurons
    layers.Dropout(0.5),
    layers.Dense(1,activation='sigmoid')
])

model.compile(optimizer = 'adam', loss='binary_crossentropy',metrics=['accuracy'])
model.fit(X_train_tfidf.toarray(),y_train,epochs=5,batch_size=64,validation_split = 0.1)


## Comprehenship Report:

```
This report will give you a coprehensive idea about what i have done in this whole project;

In this small_project i have followed certain steps, they are:

step-1: This shows details about amazonrevies.tsv dataset file like, is there is any missing value in the dataset or any duplicate values is there or not. Also in that section the uniqueness of the output has been checked.

step-2: In this step certain library used to process our text data like nltk in which there are some packages that helps in removing the punctuatuions and also some regular words(using stop words).

step-3: This step includes EDA, where libraries like wordcloud and collections being used to show the most frequent words.And also top 20 words in case of negative output and for positive output we can see clearly.This section gives more insights of words that's being used in the both the responses.

step-4: It involves in feature extraction. Here, we converted our output to 1(pos) and 0(neg) cause machine only understands numerical values. Also we uses certain methods like TF_IDF(Term frequency and inverse document frequency)  to change the words to the numerical values.

step-5: In this  portion we started building our models like LogisticRegression model, svm and neural networks and also calculation of accuracy_score and  classification_report  has been done after every model so anyone can esaily grasp.
```